In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

current = Path.cwd()

for path in [current, *current.parents]:
    if (path / "ingestion" / "delta_state.py").exists():
        sys.path.insert(0, str(path))
        print("Using import root:", path)
        break
else:
    raise RuntimeError("Could not find ingestion/delta_state.py")

In [0]:
import importlib
from datetime import date

import ingestion.delta_state as delta_state

importlib.reload(delta_state)

from src.data_generator.daily_generation import (
    generate_daily_records,
    validate_daily_records,
    daily_seed,
)

from src.data_generator.utils import initialize_randomness


SIMULATION_DATE = date(2026, 8, 23)
SEED = 42

In [0]:
state = delta_state.load_generator_state(spark)

print("users:", len(state["users"]))
print("user_state:", len(state["user_state"]))
print("models:", len(state["models"]))
print("devices:", len(state["devices"]))
print("plans:", len(state["plans"]))
print("recent conversations:", len(state["recent"]["conversations"]))
print("recent messages:", len(state["recent"]["messages"]))
print("subscriptions:", len(state["subscriptions"]))
print("count entities:", sorted(state["counts"].keys()))

In [0]:
print("user signup tz:", state["users"][0]["signup_at"].tzinfo)
print(
    "message created tz:",
    state["recent"]["messages"][0]["created_at"].tzinfo,
)
print(
    "subscription start tz:",
    state["subscriptions"][0]["started_at"].tzinfo,
)

In [0]:
import random

derived_seed = daily_seed(SIMULATION_DATE, SEED)

initialize_randomness(derived_seed)
first_numbers = [random.random() for _ in range(5)]

initialize_randomness(derived_seed)
second_numbers = [random.random() for _ in range(5)]

print("Derived seed:", derived_seed)
print("First: ", first_numbers)
print("Second:", second_numbers)
print("MATCH:", first_numbers == second_numbers)

In [0]:
state_1 = delta_state.load_generator_state(spark)

first = generate_daily_records(
    state_1,
    SIMULATION_DATE,
    SEED,
)

print("First generation complete")

for entity in [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]:
    print(f"{entity}: {len(first[entity]):,}")

In [0]:
del state_1

state_2 = delta_state.load_generator_state(spark)

print("Second state loaded fresh from Delta")

In [0]:
second = generate_daily_records(
    state_2,
    SIMULATION_DATE,
    SEED,
)

print("Second generation complete")

In [0]:
ENTITIES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

all_match = True

for entity in ENTITIES:
    match = first[entity] == second[entity]

    print(
        f"{entity}: "
        f"{len(first[entity]):,} / "
        f"{len(second[entity]):,} "
        f"{'MATCH' if match else 'DIFFERENT'}"
    )

    if not match:
        all_match = False

print("\nALL MATCH:", all_match)

In [0]:
first_validation = validate_daily_records(
    first,
    state,
    SIMULATION_DATE,
)

second_validation = validate_daily_records(
    second,
    state,
    SIMULATION_DATE,
)

print("First validation:", first_validation)
print("Second validation:", second_validation)

In [0]:
import random

import src.data_generator.daily_generation as dg

original_generate_inference_records = dg.generate_inference_records

rng_boundaries = []


def capture_rng_boundaries(messages, models, generation_end):
    before = random.getstate()

    result = original_generate_inference_records(
        messages,
        models,
        generation_end,
    )

    after = random.getstate()

    rng_boundaries.append({
        "before": before,
        "after": after,
    })

    return result


dg.generate_inference_records = capture_rng_boundaries

In [0]:
state_a = delta_state.load_generator_state(spark)

run_a = dg.generate_daily_records(
    state_a,
    SIMULATION_DATE,
    SEED,
)

del state_a


state_b = delta_state.load_generator_state(spark)

run_b = dg.generate_daily_records(
    state_b,
    SIMULATION_DATE,
    SEED,
)

del state_b

In [0]:
print("Captured:", len(rng_boundaries))

print(
    "RNG BEFORE inference match:",
    rng_boundaries[0]["before"] == rng_boundaries[1]["before"]
)

print(
    "RNG AFTER inference match:",
    rng_boundaries[0]["after"] == rng_boundaries[1]["after"]
)

print(
    "Completions match:",
    run_a["completions"] == run_b["completions"]
)

print(
    "Inferences match:",
    run_a["model_inferences"] == run_b["model_inferences"]
)